In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import curve_fit


# 1. Define the Gompertz function
def gompertz(x, a, b, c):
  return a * np.exp(-b * np.exp(-c * x))


# 2. Load and clean the ThingSpeak CSV
df = pd.read_csv("datasets/sample_3.csv")
df = df.dropna(subset=["created_at", "field1"])

# Parse ISO timestamps and sort chronologically
df["created_at"] = pd.to_datetime(df["created_at"])
df = df.sort_values("created_at")

# Convert timestamps to elapsed minutes starting at 0 for stable curve fitting
t_start = df["created_at"].min()
x_minutes = (df["created_at"] - t_start).dt.total_seconds().to_numpy() / 60.0
relative_height = df["field1"].to_numpy()

# 3. Estimate initial guesses (p0) and fit the curve
# Since x is in minutes, c_init is set to 0.05 (per minute)
p0 = [np.max(relative_height), 1.5, 0.05]
popt, pcov = curve_fit(
    gompertz, x_minutes, relative_height, p0=p0, bounds=(0, np.inf), maxfev=10000
)
a_fit, b_fit, c_fit = popt

# 4. Calculate R-squared
residuals = relative_height - gompertz(x_minutes, *popt)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((relative_height - np.mean(relative_height)) ** 2)
r_squared = 1 - (ss_res / ss_tot)

print(f"Fitted parameters: a={a_fit:.3f}, b={b_fit:.3f}, c={c_fit:.4f}")
print(f"R-squared: {r_squared:.4f}")

# 5. Generate smooth curve and convert minutes back to timestamps for Plotly
x_smooth_min = np.linspace(0, x_minutes.max(), 200)
y_smooth = gompertz(x_smooth_min, *popt)
time_smooth = t_start + pd.to_timedelta(x_smooth_min, unit="m")

# 6. Build the interactive Plotly figure
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["created_at"],
        y=relative_height,
        mode="markers",
        name="Relative Height (Observed)",
        marker=dict(color="rgba(128, 128, 128, 0.7)", size=8),
    )
)

fig.add_trace(
    go.Scatter(
        x=time_smooth,
        y=y_smooth,
        mode="lines",
        name=f"Gompertz Fit (R² = {r_squared:.4f})",
        line=dict(color="#1f77b4", width=3),
    )
)

fig.add_hline(
    y=a_fit,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Asymptote (a = {a_fit:.2f})",
    annotation_position="bottom right",
)

fig.update_layout(
    title=(
        f"Gompertz Fit for Relative Height (a={a_fit:.2f}, b={b_fit:.2f}, c={c_fit:.4f})"
    ),
    xaxis_title="Timestamp",
    yaxis_title="Relative Height [mm]",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()


In [ ]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

CSV_FILE = "data_sets/sample_2.csv"

TIMEZONE = "Europe/Berlin"

# ------------------------------------------------------------
# Phase transition timestamps
# Adjust these values after inspecting the graph.
# Format: YYYY-MM-DD HH:MM:SS
# ------------------------------------------------------------

PRE_PLATEAU_END = "2026-11-13 04:30:00"
RAMP_END = "2026-11-13 18:00:00"

# Set to None if collapse has not been observed.
COLLAPSE_START = None

# Example:
# COLLAPSE_START = "2026-11-13 21:00:00"


# ============================================================
# 2. LOAD AND PREPARE DATA
# ============================================================

df = pd.read_csv(CSV_FILE)

# Convert ThingSpeak timestamps from UTC to local time.
df["created_at"] = (
    pd.to_datetime(df["created_at"], utc=True)
    .dt.tz_convert(TIMEZONE)
)

# Extract height.
df["height"] = pd.to_numeric(
    df["field1"], errors="coerce"
)

# Remove invalid measurements and sort chronologically.
df = (
    df.dropna(subset=["created_at", "height"])
      .sort_values("created_at")
      .reset_index(drop=True)
)

if df.empty:
    raise ValueError("No valid height measurements found.")

# Experiment boundaries.
experiment_start = df["created_at"].iloc[0]
experiment_end = df["created_at"].iloc[-1]

relative_height_min = np.min(df['height'])
relative_height_max = np.max(df['height'])
max_growth = relative_height_max-relative_height_min

print(f"Minimum relative height: {relative_height_min:.2f} mm")
print(f"Maximum relative height: {relative_height_max:.2f} mm")
print(f"Total absloute growth: {max_growth:.2f} mm")

'''
We define three phases:
1. Pre-fermentation phase: happens at the beginning of fermentation and it is identified by a small or no growth in the first few hours. We define it empirically to be 5% of the total absolute growth
2. Fermentation phase: the growth is mostly linear 
3. Full fermentation phase: the growth stays constant or even drops due to dough collapse 
'''

THRESHOLD_PRE_FERMENTATION_END = 0.05 * max_growth
for idx, height in enumerate(df['height']):
    if height > THRESHOLD_PRE_FERMENTATION_END:
        print(f"{df["created_at"][idx]}")
        break


# Convert manually configured transitions to timezone-aware timestamps.
def local_timestamp(value):
    if value is None:
        return None

    timestamp = pd.Timestamp(value)

    if timestamp.tzinfo is None:
        return timestamp.tz_localize(TIMEZONE)

    return timestamp.tz_convert(TIMEZONE)


pre_plateau_end = local_timestamp(PRE_PLATEAU_END)
ramp_end = local_timestamp(RAMP_END)
collapse_start = local_timestamp(COLLAPSE_START)

